# E06 — Retrieval Optimisation — Analysis

**Question**: Which retrieval configuration most reliably retrieves ContractNLI gold evidence
while keeping the returned context compact enough for downstream classification?

Local-only, zero LLM calls throughout. Loads already-saved `results/run_E06_*.json` (produced
by `scripts/run_e06_retrieval.py`) and `results/retrieval_failure_analysis.csv` (produced by
`scripts/analyze_e06_failures.py`).

In [1]:
import csv
import json
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve().parents[1] if Path.cwd().name == "E06_retrieval_optimisation" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))
RESULTS = REPO_ROOT / "experiments/E06_retrieval_optimisation/results"

from evaluation.retrieval_eval import build_evidence_bearing_train_cases

def load(run_id):
    return json.load(open(RESULTS / f"run_E06_{run_id}.json"))

runs = {rid: load(rid) for rid in ["R0", "R1", "R2_clause256", "R2_fixed512", "R2_sentence",
                                     "R3_k3", "R3_k10", "R4_bge", "R0_matched_clause256"]}
runs["R2_clause512"] = load("R1")  # R1's config IS R2's clause-512 candidate, reused not rerun
runs["R3_k5"] = load("R2_clause256")  # R2's winning-chunking-at-K=5 IS R3's K=5 point
print("loaded", len(runs), "run results")

loaded 11 run results


## 2. Evidence-bearing TRAIN universe

In [2]:
print("n_documents:", runs['R0']['n_documents'], "n_cases:", runs['R0']['n_cases'])
print("(Entailment 3,530 + Contradiction 841 = 4,371, independently re-verified in Stage A)")

n_documents: 423 n_cases: 4371
(Entailment 3,530 + Contradiction 841 = 4,371, independently re-verified in Stage A)


## 3. Existing retrieval architecture (audited in Stage A)

`pipeline/chunker.py` (fixed/clause/sentence, unit = tiktoken tokens), `pipeline/embedder.py`
(local sentence-transformers), `pipeline/indexer.py` (FAISS IndexFlatIP, one index PER
document -- NDA-local search verified structurally), `pipeline/sparse_retriever.py` (BM25,
already bug-fixed). All reused as-is, no duplicate retriever written for E06.

## 4. R0 vs R1 — lexical (BM25) vs dense baseline

In [3]:
def row(name, r):
    m = r["metrics"]["overall"]
    c = r["metrics"]["contradiction"]
    print(f"{name:20s} recall={m['evidence_recall_at_k']:.1%}  contradiction_recall={c['evidence_recall_at_k']:.1%}  "
          f"precision={m['evidence_precision']:.1%}  mrr={m['evidence_recall_at_k'] and r['metrics']['overall']['mrr']:.3f}  "
          f"mean_tok={r['context_size']['mean_tokens']:.0f}  miss={r['metrics']['miss_count']}")

row("R0 BM25", runs["R0"])
row("R1 dense(mpnet)", runs["R1"])
print()
print("Delta R0->R1: recall", f"{runs['R1']['metrics']['overall']['evidence_recall_at_k']-runs['R0']['metrics']['overall']['evidence_recall_at_k']:+.1%}",
      " mrr", f"{runs['R1']['metrics']['overall']['mrr']-runs['R0']['metrics']['overall']['mrr']:+.3f}")
print("Finding: at clause-512/K=5, BM25 slightly BEATS dense -- both are saturated (mean_chunks~4.2,")
print("close to K=5, so top-5 returns nearly the whole document regardless of method). The method")
print("comparison is UNINFORMATIVE at this chunk size -- motivates moving to finer chunking (R2).")

R0 BM25              recall=97.8%  contradiction_recall=97.9%  precision=3.1%  mrr=0.197  mean_tok=1769  miss=47
R1 dense(mpnet)      recall=96.6%  contradiction_recall=97.8%  precision=3.1%  mrr=0.182  mean_tok=1741  miss=92

Delta R0->R1: recall -1.2%  mrr -0.015
Finding: at clause-512/K=5, BM25 slightly BEATS dense -- both are saturated (mean_chunks~4.2,
close to K=5, so top-5 returns nearly the whole document regardless of method). The method
comparison is UNINFORMATIVE at this chunk size -- motivates moving to finer chunking (R2).


## 5. R2 — chunking ladder (method=dense/mpnet, K=5 fixed)

In [4]:
for name, rid in [("clause_512 (=R1)", "R2_clause512"), ("clause_256", "R2_clause256"),
                    ("fixed_512", "R2_fixed512"), ("sentence", "R2_sentence")]:
    row(name, runs[rid])
print()
print("Semantics: clause_512/clause_256 = clause_aware_chunk (merges paragraphs up to a token")
print("budget, never cuts mid-sentence unless one sentence alone exceeds it). fixed_512 =")
print("fixed_size_chunk (token-window + 50-token overlap, no clause awareness). sentence =")
print("sentence_chunk (one chunk per sentence/clause fragment, no merging, no size parameter).")
print()
print("Finding: 512-token chunking (clause or fixed) saturates -- ~97-98% recall but MRR only")
print("~0.18, because most docs have only ~4-5 total chunks at this size (K=5 ~= 'return everything').")
print("clause_256 and sentence genuinely escape saturation, trading recall for real ranking quality.")

clause_512 (=R1)     recall=96.6%  contradiction_recall=97.8%  precision=3.1%  mrr=0.182  mean_tok=1741  miss=92
clause_256           recall=87.0%  contradiction_recall=89.7%  precision=5.0%  mrr=0.276  mean_tok=1005  miss=389
fixed_512            recall=97.8%  contradiction_recall=98.8%  precision=3.0%  mrr=0.183  mean_tok=1986  miss=51
sentence             recall=64.2%  contradiction_recall=65.5%  precision=13.8%  mrr=0.441  mean_tok=315  miss=1110

Semantics: clause_512/clause_256 = clause_aware_chunk (merges paragraphs up to a token
budget, never cuts mid-sentence unless one sentence alone exceeds it). fixed_512 =
fixed_size_chunk (token-window + 50-token overlap, no clause awareness). sentence =
sentence_chunk (one chunk per sentence/clause fragment, no merging, no size parameter).

Finding: 512-token chunking (clause or fixed) saturates -- ~97-98% recall but MRR only
~0.18, because most docs have only ~4-5 total chunks at this size (K=5 ~= 'return everything').
clause_256 and sen

## 6. R3 — top-K sweep on clause_256 (cache reused, not rebuilt)

In [5]:
for k, rid in [(3, "R3_k3"), (5, "R3_k5"), (10, "R3_k10")]:
    r = runs[rid]
    m = r["metrics"]["overall"]; c = r["metrics"]["contradiction"]
    print(f"K={k:2d}  recall={m['evidence_recall_at_k']:.1%}  contradiction_recall={c['evidence_recall_at_k']:.1%}  "
          f"precision={m['evidence_precision']:.1%}  mrr={m['mrr']:.3f}  mean_tok={r['context_size']['mean_tokens']:.0f}")

print()
k3, k5, k10 = runs["R3_k3"]["metrics"], runs["R3_k5"]["metrics"], runs["R3_k10"]["metrics"]
print(f"K3->K5:  recall {k5['overall']['evidence_recall_at_k']-k3['overall']['evidence_recall_at_k']:+.1%}, "
      f"contradiction {k5['contradiction']['evidence_recall_at_k']-k3['contradiction']['evidence_recall_at_k']:+.1%}, "
      f"mrr {k5['overall']['mrr']-k3['overall']['mrr']:+.3f}")
print(f"K5->K10: recall {k10['overall']['evidence_recall_at_k']-k5['overall']['evidence_recall_at_k']:+.1%}, "
      f"contradiction {k10['contradiction']['evidence_recall_at_k']-k5['contradiction']['evidence_recall_at_k']:+.1%}, "
      f"mrr {k10['overall']['mrr']-k5['overall']['mrr']:+.3f}")
print()
print("MRR is essentially FLAT across the whole sweep (0.271 -> 0.276 -> 0.278) -- extra K buys")
print("coverage/recall, not ranking quality. K=5 selected: captures most of the K3->K10 recall gain")
print("while keeping context far more compact than K=10 (1005 vs 1722 mean tokens) and MRR shows")
print("no benefit from going higher.")

K= 3  recall=73.2%  contradiction_recall=75.8%  precision=7.0%  mrr=0.271  mean_tok=619
K= 5  recall=87.0%  contradiction_recall=89.7%  precision=5.0%  mrr=0.276  mean_tok=1005
K=10  recall=97.5%  contradiction_recall=98.6%  precision=3.2%  mrr=0.278  mean_tok=1722

K3->K5:  recall +13.8%, contradiction +13.9%, mrr +0.005
K5->K10: recall +10.5%, contradiction +8.8%, mrr +0.002

MRR is essentially FLAT across the whole sweep (0.271 -> 0.276 -> 0.278) -- extra K buys
coverage/recall, not ranking quality. K=5 selected: captures most of the K3->K10 recall gain
while keeping context far more compact than K=10 (1005 vs 1722 mean tokens) and MRR shows
no benefit from going higher.


## 7. R4 — embedding comparison (clause_256, K=5 frozen)

In [6]:
row("mpnet (=R2_clause256)", runs["R2_clause256"])
row("bge-base-en-v1.5", runs["R4_bge"])
print()
mp, bge = runs["R2_clause256"]["metrics"], runs["R4_bge"]["metrics"]
print(f"Delta mpnet->bge: recall {bge['overall']['evidence_recall_at_k']-mp['overall']['evidence_recall_at_k']:+.1%}, "
      f"contradiction {bge['contradiction']['evidence_recall_at_k']-mp['contradiction']['evidence_recall_at_k']:+.1%}, "
      f"mrr {bge['overall']['mrr']-mp['overall']['mrr']:+.3f}")
print()
print("Real finding: bge-base-en-v1.5 gives a MEANINGFULLY better MRR (+0.034, ~12% relative)")
print("with comparable recall/precision. This CONTRADICTS the historical (post-reranking) finding")
print("that embedding model 'barely matters' -- that finding was measured with a reranker already")
print("in the pipeline; here, pre-reranking, embedding choice genuinely matters. bge-base-en-v1.5 selected.")

mpnet (=R2_clause256) recall=87.0%  contradiction_recall=89.7%  precision=5.0%  mrr=0.276  mean_tok=1005  miss=389
bge-base-en-v1.5     recall=88.4%  contradiction_recall=88.8%  precision=5.3%  mrr=0.310  mean_tok=1009  miss=349

Delta mpnet->bge: recall +1.4%, contradiction -1.0%, mrr +0.034

Real finding: bge-base-en-v1.5 gives a MEANINGFULLY better MRR (+0.034, ~12% relative)
with comparable recall/precision. This CONTRADICTS the historical (post-reranking) finding
that embedding model 'barely matters' -- that finding was measured with a reranker already
in the pipeline; here, pre-reranking, embedding choice genuinely matters. bge-base-en-v1.5 selected.


## 7b. Matched lexical control at the selected operating point (clause_256, K=5)

The original R0-vs-R1 comparison used clause_512/K=5, which the results themselves showed was
saturated (~4-5 chunks/doc, K=5 returns nearly the whole document) -- an uninformative
comparison. This control repeats BM25 vs dense at the actually-selected clause_256/K=5 point.

In [7]:
bm25_matched = load("R0_matched_clause256")
row("BM25 (clause_256, K=5)", bm25_matched)
row("dense bge (clause_256, K=5)", runs["R4_bge"])
print()
bm, dn = bm25_matched["metrics"], runs["R4_bge"]["metrics"]
print(f"Delta BM25->dense: recall {dn['overall']['evidence_recall_at_k']-bm['overall']['evidence_recall_at_k']:+.1%}, "
      f"contradiction {dn['contradiction']['evidence_recall_at_k']-bm['contradiction']['evidence_recall_at_k']:+.1%}, "
      f"precision {dn['overall']['evidence_precision']-bm['overall']['evidence_precision']:+.2%}, "
      f"mrr {dn['overall']['mrr']-bm['overall']['mrr']:+.3f}, "
      f"miss_count {dn['miss_count']-bm['miss_count']:+d}")
print()
print("REAL FINDING: at the matched, non-saturated operating point, BM25 slightly BEATS dense on")
print("overall recall (+2.1pt for BM25) and MRR (+0.012 for BM25), with FEWER misses (253 vs 349).")
print("Dense only edges ahead on Contradiction recall (+1.3pt) and precision (~tied). This genuinely")
print("challenges whether dense retrieval earns its complexity for this task -- legal/NDA hypothesis")
print("wording often literally overlaps the source clause's terminology, favoring lexical matching.")

BM25 (clause_256, K=5) recall=90.5%  contradiction_recall=87.5%  precision=5.1%  mrr=0.322  mean_tok=1038  miss=253
dense bge (clause_256, K=5) recall=88.4%  contradiction_recall=88.8%  precision=5.3%  mrr=0.310  mean_tok=1009  miss=349

Delta BM25->dense: recall -2.0%, contradiction +1.3%, precision +0.21%, mrr -0.012, miss_count +96

REAL FINDING: at the matched, non-saturated operating point, BM25 slightly BEATS dense on
overall recall (+2.1pt for BM25) and MRR (+0.012 for BM25), with FEWER misses (253 vs 349).
Dense only edges ahead on Contradiction recall (+1.3pt) and precision (~tied). This genuinely
challenges whether dense retrieval earns its complexity for this task -- legal/NDA hypothesis
wording often literally overlaps the source clause's terminology, favoring lexical matching.


## 8. Failure analysis on dense bge clause_256/K=5 (before reranking)

In [8]:
summary = json.load(open(RESULTS / "failure_analysis_summary.json"))
print(json.dumps(summary, indent=2))
print()
rows = list(csv.DictReader(open(RESULTS / "retrieval_failure_analysis.csv")))
ranks = [int(r["gold_best_rank"]) for r in rows if r["gold_best_rank"] != ">50"]
ranks.sort()
print(f"gold_best_rank: median={ranks[len(ranks)//2]}, mean={sum(ranks)/len(ranks):.1f}, "
      f"max={max(ranks)}, n_with_known_rank={len(ranks)}/{len(rows)}")
buckets = {"6-10": 0, "11-20": 0, "21-50": 0}
for r in ranks:
    if r <= 10: buckets["6-10"] += 1
    elif r <= 20: buckets["11-20"] += 1
    else: buckets["21-50"] += 1
print("rank-depth distribution:", buckets)
print()
print("100% of the 349 misses are RANKING failures (gold evidence findable within top-50, just")
print("ranked below 5) -- ZERO true retrieval-absence misses. 84.5% sit at rank 6-10 -- just past")
print("the K=5 cutoff. This is exactly trigger condition A for R5 (ranking problem).")

{
  "total_misses_analyzed": 349,
  "ranking_failure_gold_beyond_top_k": 349,
  "retrieval_absence_semantic_or_lexical_mismatch": 0,
  "multiple_evidence_spans_not_jointly_retrieved": 0,
  "misses_by_label": {
    "Entailment": 273,
    "Contradiction": 76
  }
}

gold_best_rank: median=7, mean=8.0, max=21, n_with_known_rank=349/349
rank-depth distribution: {'6-10': 295, '11-20': 53, '21-50': 1}

100% of the 349 misses are RANKING failures (gold evidence findable within top-50, just
ranked below 5) -- ZERO true retrieval-absence misses. 84.5% sit at rank 6-10 -- just past
the K=5 cutoff. This is exactly trigger condition A for R5 (ranking problem).


## 9. Representative pre-rerank failed examples

In [9]:
for r in rows[:3]:
    print("---")
    print("label:", r["label"], "| gold_best_rank:", r["gold_best_rank"])
    print("requirement:", r["requirement"])
    print("gold evidence:", r["gold_evidence_excerpt"][:200])
    print("top retrieved (rank 1):", r["top_retrieved_excerpt"][:200])

---
label: Entailment | gold_best_rank: 7
requirement: Confidential Information may include verbally conveyed information.
gold evidence: “Confidential Information” under this Agreement consists of: (iv) all observations of equipment (including computer screens) and oral disclosures related to the development of any Cyber Mutual Assista
top retrieved (rank 1): Upon receipt of a Freedom of Information Act or public records disclosure request, such Participating Entity shall: (i) notify each Participating Entity or Participating Entities whose information is 
---
label: Entailment | gold_best_rank: 7
requirement: Agreement shall not grant Receiving Party any right to Confidential Information.
gold evidence: The Data Recipient hereby acknowledges that the DOHMH is the exclusive owner of the Data and all trade secrets and other rights therein.  No license or conveyance of any such rights is granted or impl
top retrieved (rank 1): Nothing contained herein shall constitute any representation

## 10. R5 — controlled reranking comparison (APPROVED AND RUN)

**CONTROL**: dense top-20 candidates (clause_256, bge-base-en-v1.5) -> original top-5.
**RERANK**: the SAME exact top-20 candidates -> cross-encoder rerank (`ms-marco-MiniLM-L-12-v2`,
already cached) -> reranked top-5. Candidate pool identical between arms -- only the final
re-ordering/truncation differs. Pool size = 20, chosen because 84.5% of misses sat at rank
6-10 and only 1 of 349 misses was ever beyond rank 20 (deep-rank check up to 50).

In [10]:
r5 = json.load(open(RESULTS / "run_E06_R5_rerank_comparison.json"))
c, r = r5["control"]["metrics"], r5["rerank"]["metrics"]
print(f"{'CONTROL (no rerank)':22s} recall={c['overall']['evidence_recall_at_k']:.1%}  "
      f"contradiction={c['contradiction']['evidence_recall_at_k']:.1%}  "
      f"precision={c['overall']['evidence_precision']:.1%}  mrr={c['overall']['mrr']:.3f}  miss={c['miss_count']}")
print(f"{'RERANKED':22s} recall={r['overall']['evidence_recall_at_k']:.1%}  "
      f"contradiction={r['contradiction']['evidence_recall_at_k']:.1%}  "
      f"precision={r['overall']['evidence_precision']:.1%}  mrr={r['overall']['mrr']:.3f}  miss={r['miss_count']}")
print()
print(f"Recovered: {r5['recovered_count']}  Regressed: {r5['regressed_count']}  Net gain: {r5['net_gain']}")
print(f"Contradiction recovered: {r5['contradiction_recovered']}  Contradiction regressed: {r5['contradiction_regressed']}")
print()
print(f"Candidate-gen latency: mean {r5['candidate_generation_latency_ms']['mean']:.2f}ms")
print(f"Reranking latency: mean {r5['reranking_latency_ms']['mean']:.1f}ms (p90 {r5['reranking_latency_ms']['p90']:.1f}ms)")
print(f"Total query latency: mean {r5['total_query_latency_ms']['mean']:.1f}ms")
print()
print("DECISIVE WIN across every priority metric: Contradiction Recall +5.2pt (53 recovered vs 9")
print("regressed, ~6:1), overall Recall +3.8pt, MRR +0.066 (~21% relative), Precision flat/+0.08pt,")
print("for ~122ms/query added latency (negligible in absolute terms, $0 cost, local). This EARNS")
print("its complexity -- not a marginal win, decisive on the metric that matters most.")

CONTROL (no rerank)    recall=88.4%  contradiction=88.8%  precision=5.3%  mrr=0.310  miss=349
RERANKED               recall=92.2%  contradiction=93.9%  precision=5.3%  mrr=0.376  miss=190

Recovered: 231  Regressed: 72  Net gain: 159
Contradiction recovered: 53  Contradiction regressed: 9

Candidate-gen latency: mean 1.11ms
Reranking latency: mean 120.6ms (p90 217.4ms)
Total query latency: mean 121.7ms

DECISIVE WIN across every priority metric: Contradiction Recall +5.2pt (53 recovered vs 9
regressed, ~6:1), overall Recall +3.8pt, MRR +0.066 (~21% relative), Precision flat/+0.08pt,
for ~122ms/query added latency (negligible in absolute terms, $0 cost, local). This EARNS
its complexity -- not a marginal win, decisive on the metric that matters most.


## 10b. Post-rerank failure breakdown -- recovered / regressed / still unsolved

In [11]:
outcomes = [json.loads(l) for l in open(RESULTS / "run_E06_R5_per_case_outcomes.jsonl")]
still_miss = {o["case_id"] for o in outcomes if not o["rerank_hit"]}
old_failures = {row["case_id"]: row for row in rows}  # the pre-rerank 349-miss failure CSV

within_20_still_wrong = sum(1 for cid in still_miss if cid in old_failures
                             and old_failures[cid]["gold_best_rank"] != ">50"
                             and int(old_failures[cid]["gold_best_rank"]) <= 20)
beyond_pool = sum(1 for cid in still_miss if cid in old_failures
                   and (old_failures[cid]["gold_best_rank"] == ">50"
                        or int(old_failures[cid]["gold_best_rank"]) > 20))
new_regressions = sum(1 for cid in still_miss if cid not in old_failures)

print(f"Still missing after rerank: {len(still_miss)} total")
print(f"  - in candidate pool (rank<=20) but cross-encoder still didn't surface it: {within_20_still_wrong}")
print(f"  - beyond the top-20 pool entirely (a coverage ceiling, not a reranker failure): {beyond_pool}")
print(f"  - NEW regressions (control was a hit, rerank made it a miss): {new_regressions}")
print()
print("Remaining failure families: mostly genuine cross-encoder ranking limitations (117/190,")
print("~62%) where the correct chunk was visible but still not surfaced -- a real, honest residual")
print("limitation, not eliminated by reranking. Only 1 case is a true coverage gap beyond the pool.")

Still missing after rerank: 190 total
  - in candidate pool (rank<=20) but cross-encoder still didn't surface it: 117
  - beyond the top-20 pool entirely (a coverage ceiling, not a reranker failure): 1
  - NEW regressions (control was a hit, rerank made it a miss): 72

Remaining failure families: mostly genuine cross-encoder ranking limitations (117/190,
~62%) where the correct chunk was visible but still not surfaced -- a real, honest residual
limitation, not eliminated by reranking. Only 1 case is a true coverage gap beyond the pool.


## 10c. Representative reranking fix / harm / still-unsolved examples

In [12]:
recovered_ex = next(o for o in outcomes if o["recovered"])
regressed_ex = next(o for o in outcomes if o["regressed"])
unsolved_ex = next(o for o in outcomes if not o["rerank_hit"] and not o["regressed"])
manifest_cases = {c["case_id"]: c for c in build_evidence_bearing_train_cases()}

for label, o in [("RERANKING FIX", recovered_ex), ("RERANKING HARM", regressed_ex),
                  ("STILL UNSOLVED", unsolved_ex)]:
    c = manifest_cases[o["case_id"]]
    print(f"--- {label}: {o['case_id']} ({o['gold_label']}) ---")
    print("requirement:", c["hypothesis_text"][:150])
    print()

--- RERANKING FIX: train::89::nda-15 (Entailment) ---
requirement: Agreement shall not grant Receiving Party any right to Confidential Information.

--- RERANKING HARM: train::94::nda-19 (Entailment) ---
requirement: Some obligations of Agreement may survive termination of Agreement.

--- STILL UNSOLVED: train::87::nda-3 (Entailment) ---
requirement: Confidential Information may include verbally conveyed information.



## 11. Final matched control — LEXICAL+RERANK vs DENSE+RERANK

The §7b control showed BM25 was competitive with plain dense retrieval BEFORE reranking. This
final check asks the sharper question: once a reranker is already in the pipeline (as R5
established it should be), does the dense candidate GENERATOR still earn its complexity over
free BM25 candidate generation? Both arms: clause_256, top-20 candidates, same cross-encoder,
final top-5, identical 4,371-case universe -- the ONLY changed capability is BM25 vs. dense
candidate generation feeding the SAME reranker.

In [13]:
lvd = json.load(open(RESULTS / "run_E06_lexical_vs_dense_rerank.json"))
b, d = lvd["bm25_plus_rerank"]["metrics"], lvd["dense_plus_rerank"]["metrics"]
print(f"{'BM25+rerank':15s} recall={b['overall']['evidence_recall_at_k']:.2%}  "
      f"contradiction={b['contradiction']['evidence_recall_at_k']:.2%}  "
      f"precision={b['overall']['evidence_precision']:.2%}  mrr={b['overall']['mrr']:.4f}  miss={b['miss_count']}")
print(f"{'Dense+rerank':15s} recall={d['overall']['evidence_recall_at_k']:.2%}  "
      f"contradiction={d['contradiction']['evidence_recall_at_k']:.2%}  "
      f"precision={d['overall']['evidence_precision']:.2%}  mrr={d['overall']['mrr']:.4f}  miss={d['miss_count']}")
print()
print("Candidate-pool recall@20 (pre-rerank coverage):",
      f"BM25={lvd['bm25_pool_recall_at_20']:.2%}  dense={lvd['dense_pool_recall_at_20']:.2%}")
print()
ov = lvd["overlap_analysis"]
print(f"Case overlap: both_hit={ov['both_hit']}  both_miss={ov['both_miss']}  "
      f"bm25_only_hit={ov['bm25_only_hit']}  dense_only_hit={ov['dense_only_hit']}")
print(f"-> {ov['both_hit']+ov['both_miss']}/4371 ({(ov['both_hit']+ov['both_miss'])/4371:.2%}) IDENTICAL outcomes")
print()
lat = lvd["latency_ms"]
print(f"Candidate-gen latency: BM25 mean={lat['bm25_candidate_generation']['mean']:.3f}ms, "
      f"dense mean={lat['dense_candidate_generation']['mean']:.3f}ms (BM25 faster -- no embedding call)")
print(f"Reranking latency (same for both, driven by the reranker not the source): "
      f"BM25={lat['bm25_reranking']['mean']:.1f}ms, dense={lat['dense_reranking']['mean']:.1f}ms")
print()
print("DECISIVE, near-total TIE: 4,370/4,371 cases (99.98%) have IDENTICAL hit/miss outcomes")
print("between BM25+rerank and dense+rerank. Contradiction outcomes are 100% identical (841/841).")
print("Only 1 case differs anywhere, and BM25 wins it. This directly confirms -- now for candidate")
print("GENERATION METHOD, not just embedding choice among dense options -- that dense retrieval's")
print("complexity is NOT earned once a reranker is already doing the real discriminating work.")

BM25+rerank     recall=92.24%  contradiction=93.94%  precision=5.35%  mrr=0.3765  miss=189
Dense+rerank    recall=92.22%  contradiction=93.94%  precision=5.35%  mrr=0.3764  miss=190

Candidate-pool recall@20 (pre-rerank coverage): BM25=99.87%  dense=99.91%

Case overlap: both_hit=4181  both_miss=189  bm25_only_hit=1  dense_only_hit=0
-> 4370/4371 (99.98%) IDENTICAL outcomes

Candidate-gen latency: BM25 mean=0.158ms, dense mean=1.099ms (BM25 faster -- no embedding call)
Reranking latency (same for both, driven by the reranker not the source): BM25=175.5ms, dense=175.0ms

DECISIVE, near-total TIE: 4,370/4,371 cases (99.98%) have IDENTICAL hit/miss outcomes
between BM25+rerank and dense+rerank. Contradiction outcomes are 100% identical (841/841).
Only 1 case differs anywhere, and BM25 wins it. This directly confirms -- now for candidate
GENERATION METHOD, not just embedding choice among dense options -- that dense retrieval's
complexity is NOT earned once a reranker is already doing the r

## 12. Final retrieval_v1 decision — BM25+rerank selected (tie broken by simplicity)

Per the predeclared selection priority (Contradiction Recall > overall Recall > MRR >
Precision > context size > latency > simplicity) AND the explicit tie-breaking rule ("if BM25 +
reranker is effectively tied with or better than dense + reranker, prefer BM25 + reranker
because it is simpler and avoids embedding/index complexity"):

**BM25+rerank is selected.** The two arms are statistically indistinguishable (4,370/4,371
identical outcomes, Contradiction 100% identical) — this is a tie, not a marginal dense win, so
the tie-breaker applies. `retrieval_v1` is FROZEN as:

```
method:            bm25 (no embedding model, no vector index)
chunk_method:      clause
chunk_size:        256
chunk_overlap:     50
candidate_pool:    20
final_top_k:       5
reranking:         True (cross-encoder/ms-marco-MiniLM-L-12-v2)
```

Final metrics: recall 92.2%, **Contradiction Recall 93.9%**, precision 5.4%, MRR 0.376, mean
context 1,023 tokens, mean total query latency ~176ms (candidate-gen + reranking, all local, $0
cost). **This is simpler than the dense pipeline it ties**: no embedding model to load/serve, no
FAISS/vector index to build or maintain -- BM25 + the existing cross-encoder reranker is the
entire retrieval stack.

**What this means for the earlier R4 finding**: R4 found embedding choice (mpnet vs. bge)
genuinely mattered *before* reranking. This final check shows candidate-generation *method*
(lexical vs. dense) stops mattering *after* reranking is added -- both findings are real and
not contradictory: pre-reranking, the candidate generator IS the whole system, so its quality
matters; post-reranking, the cross-encoder does the real discriminating work regardless of
where the initial candidates came from, as long as the pool has reasonable coverage (both BM25
and dense hit ~99.9% pool-recall@20 here).

## 13. E03 retrieved-context generation (regenerated from TRUE FINAL retrieval_v1)

In [14]:
e03_context = json.load(open(REPO_ROOT / "experiments/E03_prompt_selection/TRAIN_PROMPT_v1_RETRIEVED_retrieval_v1.json"))
print("total cases:", e03_context["total_cases"], "| config:", e03_context["retrieval_config"])
case0 = e03_context["cases"][0]
print("model-facing fields (no gold info):", list(case0.keys()))
import statistics, tiktoken
enc = tiktoken.get_encoding("cl100k_base")
tok_lens = [len(enc.encode(" ".join(c["ranked_chunk_text"]))) for c in e03_context["cases"]]
print(f"context size for E03: mean={statistics.mean(tok_lens):.0f} median={statistics.median(tok_lens):.0f} "
      f"max={max(tok_lens)} tokens -- vs ~2,300 tokens under E03's earlier full-context condition")
print()
print("Verified: 150/150 cases present, config matches final retrieval_v1 (reranking=True),")
print("zero gold fields in any model-facing case record.")

total cases: 150 | config: {'method': 'bm25', 'chunk_method': 'clause', 'chunk_size': 256, 'chunk_overlap': 50, 'embedding_model': None, 'candidate_pool_size': 20, 'top_k': 5, 'reranking': True, 'reranker_model': 'cross-encoder/ms-marco-MiniLM-L-12-v2'}
model-facing fields (no gold info): ['case_id', 'document_id', 'hypothesis_id', 'hypothesis_text', 'retrieval_config_version', 'ranked_chunk_ids', 'ranked_chunk_text', 'ranked_chunk_offsets']
context size for E03: mean=1018 median=1026 max=1649 tokens -- vs ~2,300 tokens under E03's earlier full-context condition

Verified: 150/150 cases present, config matches final retrieval_v1 (reranking=True),
zero gold fields in any model-facing case record.
